In [4]:
import pandas as pd
import numpy as np

# CHARGEMENT
df = pd.read_csv("D:\\rouge_file\\Analyse_du_performance_de_la_bourse_casablanca\\csv_clean\\Droits_RENAMED.csv")
df = df[df['Type'] != 'Type'].drop_duplicates().copy()

print(f'Shape apres nettoyage basique : {df.shape}')
print(f'Nulls avant traitement        : {df.isnull().sum().sum()}')
print()

# GROUPE 1 — INFOS STABLES DU DROIT (< 1%)
# Forward Fill + Backward Fill par Designation
# Logique : le nom, le code ISIN, le nombre de titres et la date
# sont des caracteristiques fixes d un droit. Si une valeur
# manque pour une seance, on prend celle de la seance
# precedente ou suivante pour le meme droit.
# SUPPRESSION LIGNES PARASITES RESIDUELLES
# Logique : 17 lignes ont Designation et Reference null
# avec Instruments = chiffre seul (3,5,6...) — ce sont des
# residus OCR sans aucune information exploitable.
# On les supprime avant le ffill.
avant = len(df)
df = df[df['Designation'].notna() | df['Reference'].notna()]
df = df[~df['Instruments'].str.match(r'^\d+$', na=False)]
df = df[df['Designation'].notna()]  # supprimer les 2 lignes restantes sans designation
print(f'Lignes parasites residuelles supprimees : {avant - len(df)}')
print()

df = df.sort_values(['Designation', 'Date_Seance'])

cols_ffill = ['Nombre de titres', 'Date_Seance', 'Instruments',
              'Designation', 'Statut', 'Meilleur_Demande', 'Reference']

for col in cols_ffill:
    if col in df.columns:
        avant = df[col].isnull().sum()
        df[col] = df.groupby('Designation')[col].transform(
            lambda x: x.ffill().bfill()
        )
        apres = df[col].isnull().sum()
        print(f'[FFILL] {col:25s} : {avant} nulls -> {apres} nulls restants')

print()

# GROUPE 2 — PLUS_HAUT_JANV / PLUS_BAS_JANV (79.91%)
# Remplissage depuis Reference par groupe (Designation)
# Logique : pour les droits Non Traites (NT), le Plus_Haut et
# le Plus_Bas annuels sont souvent vides car le droit ne s echange
# pas. On utilise la valeur de Reference comme approximation —
# c est le cours theorique du droit calcule par la Bourse.
# C est une approximation valide car Reference = valeur officielle.
# On prend ensuite le max et min par Designation pour l annee.
avant_h = df['Plus_Haut_Janv'].isnull().sum()
avant_b = df['Plus_Bas_Janv'].isnull().sum()

# Calculer max et min de Reference par Designation
max_ref = df.groupby('Designation')['Reference'].transform('max')
min_ref = df.groupby('Designation')['Reference'].transform('min')

df['Plus_Haut_Janv'] = df['Plus_Haut_Janv'].fillna(max_ref)
df['Plus_Bas_Janv']  = df['Plus_Bas_Janv'].fillna(min_ref)

apres_h = df['Plus_Haut_Janv'].isnull().sum()
apres_b = df['Plus_Bas_Janv'].isnull().sum()
print(f'[REF MAX] {"Plus_Haut_Janv":25s} : {avant_h} nulls -> {apres_h} nulls restants')
print(f'[REF MIN] {"Plus_Bas_Janv":25s} : {avant_b} nulls -> {apres_b} nulls restants')

print()

# GROUPE 3 — MEILLEUR_OFFRE (88.54%)
# Remplissage depuis Reference
# Logique : quand aucune offre n est disponible dans le carnet
# d ordres, la meilleure offre theorique est le cours de Reference
# publie par la Bourse. C est une approximation conservative
# qui evite de laisser un null inutile pour les calculs de spread.
avant = df['Meilleur_Offre'].isnull().sum()
df['Meilleur_Offre'] = df['Meilleur_Offre'].fillna(df['Reference'])
apres = df['Meilleur_Offre'].isnull().sum()
print(f'[REF]     {"Meilleur_Offre":25s} : {avant} nulls -> {apres} nulls restants')

print()

# GROUPE 4 — COURS DU JOUR (94.31%)
# On laisse NaN intentionnellement
# Logique : 94% des droits sont Non Traites (NT) chaque jour —
# le marche des droits est tres peu liquide par nature.
# Un cours null = aucune transaction reelle ce jour.
# Remplacer par Reference serait une erreur car Reference est
# un cours theorique, pas un cours de marche reel.
# Ces nulls seront filtres automatiquement lors des calculs de KPIs.
nb = df['Cours du jour'].isnull().sum()
print(f'[NaN CONSERVE] {"Cours du jour":20s} : {nb} nulls conserves (droit non traite — marche illiquide)')

print()

# RAPPORT FINAL
print('RAPPORT FINAL')
print(f'Shape finale         : {df.shape}')
print(f'Nulls restants total : {df.isnull().sum().sum()}')
print()
nulls_restants = df.isnull().sum()
reste = nulls_restants[nulls_restants > 0]
if len(reste) > 0:
    print('Detail nulls restants :')
    print(reste.to_string())
else:
    print('Aucun null restant !')

df.to_csv('Droits_FINAL.csv', index=False, encoding='utf-8')
print(f'\nExporte : Droits_FINAL.csv — {len(df):,} lignes')

Shape apres nettoyage basique : (7193, 13)
Nulls avant traitement        : 24756

Lignes parasites residuelles supprimees : 17

[FFILL] Nombre de titres          : 0 nulls -> 0 nulls restants
[FFILL] Date_Seance               : 0 nulls -> 0 nulls restants
[FFILL] Instruments               : 0 nulls -> 0 nulls restants
[FFILL] Designation               : 0 nulls -> 0 nulls restants
[FFILL] Statut                    : 0 nulls -> 0 nulls restants
[FFILL] Meilleur_Demande          : 0 nulls -> 0 nulls restants
[FFILL] Reference                 : 0 nulls -> 0 nulls restants

[REF MAX] Plus_Haut_Janv            : 5731 nulls -> 0 nulls restants
[REF MIN] Plus_Bas_Janv             : 5731 nulls -> 0 nulls restants

[REF]     Meilleur_Offre            : 6352 nulls -> 0 nulls restants

[NaN CONSERVE] Cours du jour        : 6767 nulls conserves (droit non traite — marche illiquide)

RAPPORT FINAL
Shape finale         : (7176, 13)
Nulls restants total : 6767

Detail nulls restants :
Cours du jour  